In [ ]:
from google.colab import userdata
import os

# 1. Install MuJoCo & IK solver
!pip install -q mujoco mink "qpsolvers[osqp]"
!pip install -q "rerun-sdk[all]==0.28.2" "imageio[ffmpeg]"
!pip install -q "transformers<5.0.0" "lerobot==0.4.3"
!pip install -q hydra-core lightning omegaconf h5py hdf5plugin
!pip install -q stable-pretraining stable-worldmodel torchcodec
!pip install -q nnsight
!pip install -q fastapi uvicorn opencv-python

# 2. Clone Repo
!git clone --recursive https://github.com/vedpatwardhan/le-probe.git

print("✅ Environment Initialized.")
exit()

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Harvest Activations

In [ ]:
!hf auth login

In [ ]:
%cd /content/le-probe/interpretability/transcoders
%env PYTHONPATH=/env/python:/content/le-probe/interpretability/transcoders:/content/le-probe/lewm:/content/le-probe/lewm/le_wm

In [ ]:
!cp /content/drive/.../gr1_reward_tuned_v2.ckpt .

In [ ]:
!python harvest_activations.py \
    --output activations_granular \
    --workers 4

In [ ]:
!python audit_harvest.py --dir activations_granular

In [ ]:
!cp -r activations_granular /content/drive/.../activations_granular

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

### Train Transcoder

In [ ]:
%cd /content/le-probe/interpretability/transcoders
%env PYTHONPATH=/env/python:/content/le-probe/interpretability/transcoders:/content/le-probe/lewm:/content/le-probe/lewm/le_wm

In [ ]:
!cp -r /content/drive/.../activations_granular .
!cp -r /content/drive/.../gr1_reward_tuned_v2.ckpt .

In [ ]:
!bash batch_train.sh

In [ ]:
!cp -r transcoder_weights_residual /content/drive/.../transcoder_weights_residual

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

### Neuronpedia Integration

In [ ]:
!hf auth login

In [ ]:
%cd /content/le-probe/interpretability/dashboard
%env PYTHONPATH=/env/python:/content/le-probe:/content/le-probe/interpretability/transcoders:/content/le-probe/interpretability/dashboard:/content/le-probe/lewm:/content/le-probe/lewm/le_wm

In [ ]:
!cp -r /content/drive/.../transcoder_weights_residual .
!cp -r /content/drive/.../activations_granular .
!cp -r /content/drive/.../gr1_reward_tuned_v2.ckpt .

In [ ]:
!python engine.py \
    --repo vedpatwardhan/gr1_pickup_grasp \
    --meta activations_granular/encoder_L0.json \
    --model gr1_reward_tuned_v2.ckpt \
    --transcoders transcoder_weights_residual \
    --min-k 10

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

### Visualize Manifold

!hf auth login

In [ ]:
%cd /content/le-probe/interpretability/manifold
%env PYTHONPATH=/env/python:/content/le-probe:/content/le-probe/interpretability/transcoders:/content/le-probe/interpretability/manifold:/content/le-probe/lewm:/content/le-probe/lewm/le_wm

In [ ]:
!cp -r /content/drive/.../gr1_reward_tuned_v2.ckpt 

In [ ]:
!python harvest_manifold.py --episodes 200

In [ ]:
!cp manifold_data.pt /content/drive/.../manifold_data.pt

In [ ]:
from google.colab import drive
drive.flush_and_unmount()